#####  **A. The HMM(Hidden Markov Model) Parameters :**

##### **States :**
- **E**: Exon  
- **5**: 5′ Splice Site  
- **I**: Intron  

##### **Alphabet(Bases) :**
`{A, C, G, T}`

##### **Transition Probabilities :**
- `Start → E`: 1.0  
- `E → E`: 0.9  
- `E → 5`: 0.1  
- `5 → I`: 1.0  
- `I → I`: 0.9  
- `I → End`: 0.1  

##### **Emission Probabilities :**

| State | A    | C    | G    | T    |
|-------|------|------|------|------|
| **E** | 0.25 | 0.25 | 0.25 | 0.25 |
| **5** | 0.05 | 0.0  | 0.95 | 0.0  |
| **I** | 0.4  | 0.1  | 0.1  | 0.4  |

##### **B. Function to calculate the Log Probability of a given Path :**

The function `get_log_prob_of_a_given_path` return the log probability of the path, sequence pair passed as its arguements
- It starts by initializing prev = Start indicating that we start from the Start state
- For each `i = (0,.....,n-1)` (n == length of the nucleotide sequence provided), it add the log of the transition probability from previous state to the current and the log of the emission probability of the nucleotide at position i from the current state to the variable `log_prob`
- Finally `log_prob` contains the final log probability of the given path 


In [2]:
import math

def log(x):
    if(x == 0):
        return -math.inf
    else:
        return math.log(x)

def get_log_prob_of_a_given_path(state_path, sequence):

    if(len(state_path) != len(sequence)):
        raise ValueError("State path and sequence must be of the same length")

    log_prob = 0.0
    prev = 'Start'
    for i in range(len(sequence)):
        s = state_path[i]
        b = sequence[i]
        log_prob = log_prob + log(transition_prob[prev][s]) + log(emission_prob[s][b])
        prev = s
    # Final transition to End (only Possible from I)
    if prev == 'I':
        log_prob += log(transition_prob[prev]['End'])
    return round(log_prob, 2)


states = ['E', '5', 'I']
transition_prob = {
    'Start': {'E': 1.0},
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9, 'End': 0.1}
}

emission_prob = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}
state_path = "EEEEEEEEEEEEEEEEEE5IIIIIII" 
sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
prob = get_log_prob_of_a_given_path(state_path, sequence)
print(f"Log probability of the given path: {prob}")

Log probability of the given path: -41.22


##### **C. Viterbi Algorithm Implementation**

**1.Initialization (i = 0)**
- Set initial probabilities for only state `'E'` (since the only valid transition from `'Start'` is to `'E'`).
- Initialize the path: `path['E'] = ['E']` which indicates that the best path ending in state `'E'` is E

**2.Dynamic Programming Loop (i = 1 to n-1)**
  
  For each `i`:
  - For each possible `curr_state`:
    - Consider all valid `prev_state` values that can transition into `curr_state`.
    - Compute the total log-probability of:
      - Reaching `curr_state` from `prev_state`, and
      - Emitting `sequence[t]` from `curr_state`.
    - Keep track of the maximum such probability and the corresponding `prev_state`.
    - Store this max log-probability in `V[t][curr_state]`.
    - Update the most probable path leading to `curr_state`.

**3.Termination (After processing the full sequence)**  
  - Among all possible ending states at time `n-1`, select the one with the **highest log-probability**.
  - This is the **final state** of the most likely path.

**4.Output**  
  - Print the most likely path: `path[final_state]`
  - Print the corresponding log-probability: `max_final_prob`

In [5]:
n = len(sequence)
V = [{}]
path = {}

# Initialization
for s in ['E']:
    V[0][s] = math.log(transition_prob['Start'][s]) + log(emission_prob[s][sequence[0]])
    path[s] = [s]
# Dynamic Programming Loop
for i in range(1, n):
    V.append({})
    new_path = {}
    for curr_state in states:
        max_prob = -math.inf
        prev_state_selected = None
        for prev_state in V[i-1]:
            if prev_state in transition_prob and curr_state in transition_prob[prev_state]:
                prob = V[i-1][prev_state] + (log(transition_prob[prev_state][curr_state])) + (log(emission_prob[curr_state][sequence[i]]))
                if prob > max_prob:
                    max_prob = prob
                    prev_state_selected = prev_state
        if prev_state_selected is not None:
            V[i][curr_state] = max_prob
            new_path[curr_state] = path[prev_state_selected] + [curr_state]
    path = new_path

# Termination
max_final_prob = float('-inf')
final_state = None
for s in states:
    prob = V[n-1][s]
    if prob > max_final_prob:
        max_final_prob = prob
        final_state = s

print("Most likely path for the given Example Sequence: ", ''.join(path[final_state]))
print("Viterbi log probability for the most likey path: ", max_final_prob)


Most likely path for the given Example Sequence:  EEEEEEEEEEEEEEEEEEEEEEEEEE
Viterbi log probability for the most likey path:  -38.677666280562796
